# Order Manager

## Overview

This notebook demonstrates the public order-recording methods in `order_manager` using a temporary SQLite state store.
- Problem: the runtime needs durable order records and a position derived from confirmed executions, while notebook experimentation must not submit a real order.
- Approach: require dry-run mode, submit a simulated order to a temporary SQLite database, and inspect the resulting order and position.
- Order Recording: It records a simulated market order without sending an exchange request.
- Position: It reads the net base-asset position from the execution store.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory


In [ ]:
from IPython.display import display

from src.live_trading.config import load_live_trading_config
from src.live_trading.kraken_client import KrakenClient
from src.live_trading.order_manager import OrderManager


## Order Recording

This cell uses a temporary state store in dry-run mode.
- `KRAKEN_DRY_RUN=true` prevents `OrderManager` from calling Kraken's order-creation endpoint.
- The temporary database keeps the demonstration separate from the configured live state store.
- A simulated order has zero filled quantity, so it does not change the calculated position.


In [ ]:
config = load_live_trading_config()
if not config.dry_run:
    raise ValueError("Set KRAKEN_DRY_RUN=true before running this notebook example.")

with TemporaryDirectory() as temporary_directory:
    client = KrakenClient(config)
    order_manager = OrderManager(
        client=client,
        state_db_path=Path(temporary_directory) / "trading_state.db",
        dry_run=config.dry_run,
    )
    order = order_manager.submit_order(
        symbol=config.symbol,
        side="buy",
        amount=config.order_size,
    )
    position = order_manager.get_position(config.symbol)

display(order)
display({"symbol": config.symbol, "net_position": position})
